In [ ]:
%pip install pandas>=2.2 networkx decorator tqdm six scipy
%pip install littleballoffur --no-deps

In [ ]:
pip install torch_geometric

In [ ]:
pip install networkit

## Greedy Graph Coreset Selection Algorithm

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from littleballoffur import RandomWalkSampler, RandomNodeSampler, DegreeBasedSampler, PageRankBasedSampler, ForestFireSampler, MetropolisHastingsRandomWalkSampler, SnowBallSampler
import random
import warnings

warnings.filterwarnings("ignore")


def OMP_sampling(L,NIt):

    n, _ = np.shape(L)
    b = np.zeros((n,1))
    x = []

    for i in range(NIt):

        IND = np.zeros((n,1))
        idx_list = random.sample(range(n), k=100)
        xa = abs(L[:,idx_list].T.dot(b))
        mx = min(xa)

        IND = (xa<=mx)

        Ind = [index for index,value in enumerate(IND) if value]
        Ind = Ind[random.randrange(len(Ind))]
        x.append(idx_list[Ind])

        b = b - L[:,[idx_list[Ind]]]
        L[:,[idx_list[Ind]]] = np.inf*np.ones((n,1))

    return x

## Testing Function

In [ ]:
from scipy.sparse.linalg import eigsh


def build_signal(f_type, n, K, L):
    """
    Build a test signal f : V -> R.

    Parameters
    ----------
    f_type  : 'smooth' | 'linear' | 'degree' | 'random'
    n       : number of nodes
    cluster : (n,) array of cluster assignments
    K       : number of clusters
    L       : (n, n) graph Laplacian (used for degree-based signal)

    Returns
    -------
    f : (n,) ndarray
    """
    if f_type == 'smooth':
        # cluster-constant: low frequency, lives in span{u_0,...,u_{K-1}}
        vals = np.arange(1, K + 1)
        f    = vals[cluster].astype(float)

    elif f_type == 'pw_random':
        # random linear combination of first K eigenvectors
        _, U = np.linalg.eigh(L.todense())
        U_K  = U[:, :K]
        #alpha = np.sqrt(n)*np.random.standard_normal(K)
        alpha = np.random.random(K)*np.sqrt(n)
        f    = U_K @ alpha

    elif f_type == 'pw_uniform':
        # uniform combination: f = sum of first K eigenvectors
        _, U = np.linalg.eigh(L.todense())
        f    = U[:, :K].sum(axis=1)

    elif f_type == 'pw_weighted':
        # frequency-weighted: emphasises lower modes
        eigvals, U = np.linalg.eigh(L.todense())
        eigvals_K  = eigvals[:K]
        alpha      = 1.0 / (eigvals_K + 1e-3)
        f          = U[:, :K] @ alpha

    elif f_type == 'linear':
        # ramps across node index: mid frequency
        f = np.arange(n, dtype=float)

    elif f_type == 'degree':
        # correlated with graph structure
        f = np.array(L.diagonal(), dtype=float)

    elif f_type == 'random':
        # pure noise: high frequency, no cluster structure
        rng = np.random.default_rng(0)
        f   = rng.standard_normal(n)

    else:
        raise ValueError(f"unknown f_type '{f_type}'")

    return f


def single_function_test(f, sample_indices, label='sample'):
    """
    Test whether sample S gives a good estimate of mean(f).

    Parameters
    ----------
    f              : (n,) signal defined on all nodes
    sample_indices : list or array of sampled node indices
    label          : name for this sampling method

    Returns
    -------
    dict with mu_true, mu_sample, error, rel_error
    """
    mu_true   = f.mean()
    mu_sample = f[sample_indices].mean()
    error     = abs(mu_sample - mu_true)
    rel_error = error / (abs(mu_true) + 1e-10)

    return {
        'mu_true'  : mu_true,
        'mu_sample': mu_sample,
        'error'    : error,
        'rel_error': rel_error,
    }

## Utility Functions

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def make_clustered_rgg(n, n_clusters, spread, radius):
    rng = np.random.default_rng()

    angles = np.linspace(0, 2 * np.pi, n_clusters, endpoint=False)
    centers = np.stack([0.5 + 0.3 * np.cos(angles),
                        0.5 + 0.3 * np.sin(angles)], axis=1)

    labels = rng.integers(0, n_clusters, size=n)

    pos_array = np.zeros((n, 2))
    for i in range(n):
        pos_array[i] = rng.normal(loc=centers[labels[i]], scale=spread)
    pos_array = np.clip(pos_array, 0, 1)

    pos_dict = {i: pos_array[i].tolist() for i in range(n)}
    G = nx.random_geometric_graph(n, radius, pos=pos_dict)

    return G, pos_dict, labels

def cluster_imbalance(labels, sampled_nodes, n_clusters):
    # expected fraction if sampling were perfectly proportional
    total        = len(labels)
    cluster_size = np.array([np.sum(labels == k) for k in range(n_clusters)])
    expected_frac = len(sampled_nodes) / total  # shape (n_clusters,)

    # actual fraction of each cluster that was sampled
    sampled_labels = labels[list(sampled_nodes)]
    sampled_size   = np.array([np.sum(sampled_labels == k) for k in range(n_clusters)])
    actual_frac    = sampled_size / cluster_size  # shape (n_clusters,)

    # imbalance: mean absolute deviation from expected
    imbalance = np.mean(np.abs(actual_frac - expected_frac))

    ## print(f"{'Mean imbalance error:':<40} {imbalance*100:.2f}%")

    return imbalance

## Coreset Visualization for Random Geometric Graph

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

num_sampled = 50

# --- build original graph ---
G = nx.random_geometric_graph(1000, radius=0.1) # seed = 42
pos = nx.get_node_attributes(G, 'pos')

# --- subsample ---
# sampled_nodes = set(np.random.choice(list(G.nodes()), size=num_sampled, replace=False))

A = nx.adjacency_matrix(G)
n, _ = np.shape(A)
L = nx.laplacian_matrix(G)
LL = nx.laplacian_matrix(G)
sampled_nodes = OMP_sampling(L,num_sampled)

# model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
# subsampled_graph = model.sample(G)
# sampled_nodes = np.asarray(subsampled_graph.nodes)

# --- node styling ---
node_color = ['#1D9E75' if n in sampled_nodes else '#D3D1C7' for n in G.nodes()]
node_size  = [40        if n in sampled_nodes else 10        for n in G.nodes()]
node_alpha = [1.0       if n in sampled_nodes else 0.4       for n in G.nodes()]

# --- edge styling: highlight edges between sampled nodes ---
edge_colors = []
edge_widths = []
for u, v in G.edges():
    if u in sampled_nodes and v in sampled_nodes:
        edge_colors.append('#1F77B4')
        edge_widths.append(1.2)
    else:
        edge_colors.append('#B4B2A9')
        edge_widths.append(0.3)

fig, ax = plt.subplots(figsize=(6, 6))

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       alpha=0.5, ax=ax)

# draw non-sampled nodes first (background)
non_sampled = [n for n in G.nodes() if n not in sampled_nodes]
nx.draw_networkx_nodes(G, pos, nodelist=non_sampled,
                       node_color='#D3D1C7', node_size=10,
                       alpha=0.4, ax=ax)

# draw sampled nodes on top
nx.draw_networkx_nodes(G, pos, nodelist=list(sampled_nodes),
                       node_color='#1F77B4', node_size=40,
                       alpha=1.0, ax=ax)

# ax.set_title(f"Original graph (n={G.number_of_nodes()})  ·  "
#              f"Highlighted subsample (k={len(sampled_nodes)})", fontsize=12)
ax.axis('off')
plt.tight_layout()

plt.show()

## Mean Estimation for Band-limited Function

In [ ]:
# --- subsample ---

iter = 10

mu_true = 0
results_omp_average_error, results_omp_average_rel_error = 0, 0
results_RN_average_error, results_RN_average_rel_error = 0, 0
results_PR_average_error, results_PR_average_rel_error = 0, 0
results_DB_average_error, results_DB_average_rel_error = 0, 0
results_MH_average_error, results_MH_average_rel_error = 0, 0
results_uniform_average_error, results_uniform_average_rel_error = 0, 0

for i in range(iter):

    # --- build original graph ---

    num_sampled = 50
    G = nx.random_geometric_graph(1000, radius=0.1)
    pos = nx.get_node_attributes(G, 'pos')

    # --- different sampling methods ---
    sampled_nodes_uniform = set(np.random.choice(list(G.nodes()), size=num_sampled, replace=False))

    A = nx.adjacency_matrix(G)
    n, _ = np.shape(A)
    L = nx.laplacian_matrix(G)
    LL = nx.laplacian_matrix(G)
    sampled_nodes_omp = OMP_sampling(L,num_sampled)

    model = RandomNodeSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_RN = np.asarray(subsampled_graph.nodes)

    model = PageRankBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_PR = np.asarray(subsampled_graph.nodes)

    model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_DB = np.asarray(subsampled_graph.nodes)

    model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_MH = np.asarray(subsampled_graph.nodes)

    #cluster = np.array(cluster, dtype=int)
    f_type = 'pw_random'

    f = build_signal(f_type, n, 3, LL)
    ## print(f"signal type: {f_type},  true mean μ = {f.mean():.3f}")
    mu_true = mu_true + abs(f.mean()/iter)

    results_omp = single_function_test(f, sampled_nodes_omp, label='OMP')
    results_RN = single_function_test(f, sampled_nodes_RN, label='Random Node')
    results_PR = single_function_test(f, sampled_nodes_PR, label='Page Rank')
    results_DB = single_function_test(f, sampled_nodes_DB, label='Degree Based')
    results_MH = single_function_test(f, sampled_nodes_MH, label='MetropolisHastings')
    results_uniform = single_function_test(f, list(sampled_nodes_uniform), label='uniform')

    results_omp_average_error = results_omp_average_error + results_omp['error']/iter
    results_omp_average_rel_error = results_omp_average_rel_error + results_omp['rel_error']/iter
    results_RN_average_error = results_RN_average_error + results_RN['error']/iter
    results_RN_average_rel_error = results_RN_average_rel_error + results_RN['rel_error']/iter
    results_PR_average_error = results_PR_average_error + results_PR['error']/iter
    results_PR_average_rel_error = results_PR_average_rel_error + results_PR['rel_error']/iter
    results_DB_average_error = results_DB_average_error + results_DB['error']/iter
    results_DB_average_rel_error = results_DB_average_rel_error + results_DB['rel_error']/iter
    results_MH_average_error = results_MH_average_error + results_MH['error']/iter
    results_MH_average_rel_error = results_MH_average_rel_error + results_MH['rel_error']/iter
    results_uniform_average_error = results_uniform_average_error + results_uniform['error']/iter
    results_uniform_average_rel_error = results_uniform_average_rel_error + results_uniform['rel_error']/iter

print(mu_true)

print(results_omp_average_error)
print(results_RN_average_error)
print(results_PR_average_error)
print(results_DB_average_error)
print(results_MH_average_error)
print(results_uniform_average_error)

## Visualization for Random Geometric Graph with Clustered Strucutre

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def make_clustered_rgg(n, n_clusters, spread, radius):
    rng = np.random.default_rng()

    angles = np.linspace(0, 2 * np.pi, n_clusters, endpoint=False)
    centers = np.stack([0.5 + 0.3 * np.cos(angles),
                        0.5 + 0.3 * np.sin(angles)], axis=1)

    labels = rng.integers(0, n_clusters, size=n)

    pos_array = np.zeros((n, 2))
    for i in range(n):
        pos_array[i] = rng.normal(loc=centers[labels[i]], scale=spread)
    pos_array = np.clip(pos_array, 0, 1)

    pos_dict = {i: pos_array[i].tolist() for i in range(n)}
    G = nx.random_geometric_graph(n, radius, pos=pos_dict)

    return G, pos_dict, labels

G, pos, labels = make_clustered_rgg(
    n=1000,
    n_clusters=5,
    spread=0.06,
    radius=0.12,
)

colors = [cm.tab10(labels[n]) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(6, 6))
nx.draw_networkx_edges(G, pos, edge_color='#B4B2A9', width=0.3, alpha=0.4, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=20, ax=ax)
ax.set_title(f"Clustered RGG  ·  {len(set(labels))} communities  ·  "
             f"n={G.number_of_nodes()}, m={G.number_of_edges()}")
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
num_sampled = 20

# rng = np.random.default_rng()
# sampled_nodes = set(rng.choice(list(G.nodes()), size=num_sampled, replace=False))

A = nx.adjacency_matrix(G)
n, _ = np.shape(A)
L = nx.laplacian_matrix(G)
LL = nx.laplacian_matrix(G)
sampled_nodes = OMP_sampling(L,num_sampled)

# model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
# subsampled_graph = model.sample(G)
# sampled_nodes = np.asarray(subsampled_graph.nodes)


non_sampled   = [n for n in G.nodes() if n not in sampled_nodes]

# --- edge styling ---
edge_colors = ['#B4B2A9' if (u in sampled_nodes and v in sampled_nodes) else '#B4B2A9'
               for u, v in G.edges()]
edge_widths  = [1.2      if (u in sampled_nodes and v in sampled_nodes) else 0.3
               for u, v in G.edges()]

# --- community colors for background nodes ---
bg_colors = [cm.tab10(labels[n]) for n in non_sampled]

fig, ax = plt.subplots(figsize=(6, 6))

nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       alpha=0.5, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=non_sampled,
                       node_color=bg_colors, node_size=15,
                       alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=list(sampled_nodes),
                       node_color='#000000', node_size=50,
                       alpha=1.0, ax=ax)

# ax.set_title(f"Clustered RGG  ·  {len(set(labels))} communities  ·  "
#              f"subsample k={len(sampled_nodes)} / n={G.number_of_nodes()}")
ax.axis('off')
plt.tight_layout()

plt.show()

## Cluster Balance Error

In [ ]:
iter = 10

imbalance_error_omp = 0
imbalance_error_RN = 0
imbalance_error_PR = 0
imbalance_error_DB = 0
imbalance_error_MH = 0
imbalance_error_uniform = 0

for i in range(iter):
    G, pos, labels = make_clustered_rgg(
        n=1000,
        n_clusters=5,
        spread=0.06,
        radius=0.12,
    )

    # --- subsample ---
    num_sampled = 50

    rng = np.random.default_rng()
    sampled_nodes_uniform = set(rng.choice(list(G.nodes()), size=num_sampled, replace=False))

    A = nx.adjacency_matrix(G)
    n, _ = np.shape(A)
    L = nx.laplacian_matrix(G)
    LL = nx.laplacian_matrix(G)
    sampled_nodes_omp = OMP_sampling(L,num_sampled)

    model = RandomNodeSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_RN = np.asarray(subsampled_graph.nodes)

    model = PageRankBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_PR = np.asarray(subsampled_graph.nodes)

    model = DegreeBasedSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_DB = np.asarray(subsampled_graph.nodes)

    model = MetropolisHastingsRandomWalkSampler(number_of_nodes=num_sampled,seed=None)
    subsampled_graph = model.sample(G)
    sampled_nodes_MH = np.asarray(subsampled_graph.nodes)

    # non_sampled   = [n for n in G.nodes() if n not in sampled_nodes]

    # --- cluster imbalance ---
    imbalance_error_omp = imbalance_error_omp + cluster_imbalance(labels, sampled_nodes_omp, n_clusters=5)/iter
    imbalance_error_RN = imbalance_error_RN + cluster_imbalance(labels, sampled_nodes_RN, n_clusters=5)/iter
    imbalance_error_PR = imbalance_error_PR + cluster_imbalance(labels, sampled_nodes_PR, n_clusters=5)/iter
    imbalance_error_DB = imbalance_error_DB + cluster_imbalance(labels, sampled_nodes_DB, n_clusters=5)/iter
    imbalance_error_MH = imbalance_error_MH + cluster_imbalance(labels, sampled_nodes_MH, n_clusters=5)/iter
    imbalance_error_uniform = imbalance_error_uniform + cluster_imbalance(labels, sampled_nodes_uniform, n_clusters=5)/iter


print(imbalance_error_omp)
print(imbalance_error_RN)
print(imbalance_error_PR)
print(imbalance_error_DB)
print(imbalance_error_MH)
print(imbalance_error_uniform)